# HalluDet-Lite Phase 5 â€” Phi-2 DPO Fine-Tuning

**Run this notebook in Google Colab with T4 GPU.**

**Runtime > Change runtime type > T4 GPU**

## What this does
Fine-tunes Microsoft Phi-2 (2.7B params) using Direct Preference Optimization (DPO)
on hallucination preference pairs from HaluEval.

The fine-tuned LoRA adapter (~40MB) is saved to Google Drive.
Download it to your PC as `models/phi2_dpo_lora/`.

## Hardware
- T4 GPU (15GB VRAM) â€” free Colab tier
- Session time: ~2 hours total
- Checkpoints every 50 steps â€” safe to resume if session expires

In [ ]:
# â”€â”€ Cell 1: Install dependencies â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
!pip install transformers==4.40.0 datasets peft==0.10.0 trl==0.8.6 bitsandbytes==0.41.3 accelerate wandb -q
print('Dependencies installed.')

In [ ]:
# â”€â”€ Cell 2: Mount Google Drive (checkpoints save here) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/halludet_phi2'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints will save to: {DRIVE_DIR}')

In [ ]:
# â”€â”€ Cell 3: Verify GPU â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')
# Expected: Tesla T4, 15.0GB

In [ ]:
# â”€â”€ Cell 4: Load dataset â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# OPTION A: Upload dpo_dataset.json from your PC (recommended)
# Use: Files panel (left sidebar) > Upload > select data/processed/dpo_dataset.json
DATA_PATH = '/content/dpo_dataset.json'

# OPTION B: If you uploaded to Drive:
# DATA_PATH = f'{DRIVE_DIR}/dpo_dataset.json'

# OPTION C: Build dataset directly in Colab (slower but doesn't need your PC file)
# Uncomment to use:
# from datasets import load_dataset
# halueval = load_dataset('pminervini/HaluEval', 'qa_samples', split='train')
# pairs = [{'prompt': s['question'], 'chosen': s['right_answer'],
#            'rejected': s['hallucinated_answer']} for s in halueval]
# import json, random; random.shuffle(pairs)
# n_val = int(len(pairs) * 0.1)
# with open(DATA_PATH, 'w') as f:
#     json.dump({'train': pairs[n_val:], 'val': pairs[:n_val]}, f)

import json
with open(DATA_PATH) as f:
    raw = json.load(f)

from datasets import Dataset
train_ds = Dataset.from_list(raw['train'])
eval_ds  = Dataset.from_list(raw['val'])
print(f'Train: {len(train_ds):,} | Val: {len(eval_ds):,}')

In [ ]:
# â”€â”€ Cell 5: Load Phi-2 with QLoRA â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL = 'microsoft/phi-2'

# 4-bit quantization â€” T4 uses fp16 (NOT bf16)
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

print('Loading Phi-2...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map='auto', trust_remote_code=True
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
tokenizer.pad_token   = tokenizer.eos_token
tokenizer.padding_side = 'left'

# LoRA â€” only 3M of 2.7B params are trainable
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'dense'],
    bias='none'
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print('Phi-2 + LoRA loaded.')

In [ ]:
# â”€â”€ Cell 6: Load reference model (frozen base) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('Loading reference model (frozen Phi-2)...')
ref_model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map='auto', trust_remote_code=True
)
print('Reference model loaded.')

In [ ]:
# â”€â”€ Cell 7: DPO Training â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from trl import DPOTrainer, DPOConfig

# Check if resuming from checkpoint
import glob
checkpoints = sorted(glob.glob(f'{DRIVE_DIR}/checkpoint-*'))
resume_from = checkpoints[-1] if checkpoints else None
if resume_from:
    print(f'Resuming from checkpoint: {resume_from}')

cfg = DPOConfig(
    output_dir=DRIVE_DIR,
    beta=0.1,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,     # effective batch = 16
    learning_rate=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    fp16=True, bf16=False,             # T4 = fp16
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    max_length=512,
    max_prompt_length=256,
    evaluation_strategy='steps', eval_steps=50,
    save_strategy='steps', save_steps=50,
    save_total_limit=3,
    logging_steps=10,
    report_to='none',                  # set to 'wandb' if you have it
    remove_unused_columns=False,
)

trainer = DPOTrainer(
    model=model, ref_model=ref_model, args=cfg,
    train_dataset=train_ds, eval_dataset=eval_ds, tokenizer=tokenizer
)

print('Starting DPO training...')
print('Watch: rewards/margins should increase steadily')
print('If kl_divergence > 5.0, increase beta in cfg')
trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
# â”€â”€ Cell 8: Save final adapter â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
final_dir = f'{DRIVE_DIR}/final'
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f'LoRA adapter saved to: {final_dir}')
print()
print('NEXT STEPS:')
print('1. In Google Drive, navigate to MyDrive/halludet_phi2/final/')
print('2. Download the entire folder')
print('3. Place it on your PC at: halludet_lite/models/phi2_dpo_lora/final/')
print('4. Run: python scripts/run_all_eval.py')

In [ ]:
# â”€â”€ Cell 9 (Optional): Quick TruthfulQA test in Colab â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('Testing fine-tuned model on sample questions...')
from peft import PeftModel

test_questions = [
    'When did Albert Einstein win the Nobel Prize?',
    'What is the capital of Australia?',
    'Who wrote Romeo and Juliet?',
]

for q in test_questions:
    prompt = f'Answer accurately: {q}\nAnswer:'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=60, temperature=0.7,
                              do_sample=True, top_p=0.9)
    new_tok = out[0][inputs['input_ids'].shape[1]:]
    answer  = tokenizer.decode(new_tok, skip_special_tokens=True)
    print(f'Q: {q}')
    print(f'A: {answer.strip()}')
    print()